# Stage 1 Validation — Credit Risk Training (`T`) & MLflow 3 Model Registry

Interactive validation notebook for testing the Feature Pipeline (`F`), Feast Offline Feature Store, PoC Model 1 (`credit_risk_batch` — LightGBM -> ONNX) Training Pipeline (`T`), and the **MLflow 3 `MlflowModelRegistry`** inside Docker (`docker compose up --build -d`) or offline:

- **Cells 1–2** inspect the shared `feast_data`, `model_artifacts`, and containerized MLflow registry (`http://localhost:5000`) populated by `feast`, `feature`, `mlflow`, and `train` without mutating shared Feast tables.
- **Cells 3–5** run an isolated end-to-end `FeaturePipeline -> FeastFeatureStore -> TrainingPipeline -> MlflowModelRegistry -> ONNX Runtime` workflow, verifying convergence, ONNX parity (`< 1e-5`), cohort fairness, artifact restoration from MLflow, `@champion` / `@previous` rotation, `feature_view_version` compatibility guards, and atomic rollback.

In [ ]:
import os
import tempfile
from datetime import UTC, datetime, timedelta
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd

from banking_fti.adapters import FeastFeatureStore, MlflowModelRegistry
from banking_fti.core.contracts import (
    HOME_OWNERSHIP_VALUES,
    RAW_CARD_EVENTS_COLUMNS,
    RawBankingBatch,
)
from banking_fti.features.pipeline import FeaturePipeline
from banking_fti.training import (
    CreditRiskGateConfig,
    TabularPreprocessor,
    TrainingPipeline,
)

repo_path = Path(os.environ.get("FEAST_REPO_PATH", "feature_repo")).resolve()
shared_default = Path(tempfile.gettempdir()) / "banking_fti_feast_shared_train"
shared_data_dir = Path(os.environ.get("FEAST_DATA_DIR", str(shared_default))).resolve()
shared_store = FeastFeatureStore(repo_path=repo_path, data_dir=shared_data_dir)

shared_artifact_default = Path(tempfile.gettempdir()) / "banking_fti_model_artifacts_shared"
shared_artifact_dir = Path(
    os.environ.get("MODEL_ARTIFACT_DIR", str(shared_artifact_default))
).resolve()

sandbox_feast_default = Path(tempfile.gettempdir()) / "banking_fti_train_nb_feast"
sandbox_feast_dir = Path(
    os.environ.get("TRAINING_NOTEBOOK_SANDBOX_FEAST_DIR", str(sandbox_feast_default))
).resolve()
sandbox_store = FeastFeatureStore(repo_path=repo_path, data_dir=sandbox_feast_dir)

sandbox_art_default = Path(tempfile.gettempdir()) / "banking_fti_train_nb_artifacts"
sandbox_artifact_dir = Path(
    os.environ.get("TRAINING_NOTEBOOK_SANDBOX_ARTIFACT_DIR", str(sandbox_art_default))
).resolve()

sandbox_mlflow_env = os.environ.get("TRAINING_NOTEBOOK_SANDBOX_MLFLOW_DIR")
live_mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "").strip()
if sandbox_mlflow_env or not live_mlflow_uri:
    mlflow_dir = Path(
        sandbox_mlflow_env or str(Path(tempfile.gettempdir()) / "banking_fti_train_nb_mlflow")
    ).resolve()
    mlflow_dir.mkdir(parents=True, exist_ok=True)
    sandbox_tracking_uri = f"sqlite:///{mlflow_dir / 'mlflow.db'}"
    sandbox_model_name = "credit_risk_batch"
    sandbox_experiment = "banking_fti_nb_sandbox"
else:
    sandbox_tracking_uri = live_mlflow_uri
    sandbox_model_name = "credit_risk_batch_nb_validation"
    sandbox_experiment = "banking_fti_nb_validation"

registered_entities = sorted(entity.name for entity in shared_store.list_entities())
registered_views = sorted(view.name for view in shared_store.list_feature_views())
shared_counts = {
    group: len(shared_store.read_feature_group(group))
    for group in ("customer_profiles", "card_window_aggs", "credit_applications")
}
persisted_artifacts = (
    sorted(
        str(p.relative_to(shared_artifact_dir))
        for p in shared_artifact_dir.rglob("*")
        if p.is_file()
    )
    if shared_artifact_dir.exists()
    else []
)
print("Entities:", registered_entities)
print("FeatureViews:", registered_views)
print("Shared container row counts:", shared_counts)
print("Shared container model artifacts:", persisted_artifacts)
print("MLflow tracking URI for validation:", sandbox_tracking_uri)

if live_mlflow_uri and not sandbox_mlflow_env:
    live_registry = MlflowModelRegistry.from_env()
    for alias_name, getter in (
        ("champion", live_registry.get_champion),
        ("challenger", live_registry.get_challenger),
        ("previous", live_registry.get_previous),
    ):
        mv = getter("credit_risk_batch")
        if mv is not None:
            print(
                f"Live MLflow @{alias_name}: v{mv.version} "
                f"fv={mv.feature_view_name}:{mv.feature_view_version} "
                f"git={mv.git_sha} schema={mv.feature_schema_hash[:12]} "
                f"dataset={mv.dataset_sha256[:12]} metrics={dict(mv.metrics)} "
                f"onnx={mv.onnx_model_path}"
            )
    print("MLflow UI available at http://localhost:5000")

In [ ]:
if shared_counts["credit_applications"] > 0:
    shared_registry = MlflowModelRegistry(
        tracking_uri=sandbox_tracking_uri,
        experiment_name=f"{sandbox_experiment}_shared",
        environment="local",
        artifact_cache_dir=sandbox_artifact_dir / "shared_mlflow_cache",
    )
    shared_pipeline = TrainingPipeline(
        feature_store=shared_store,
        model_registry=shared_registry,
        model_name="credit_risk_batch_nb_shared",
        artifact_dir=sandbox_artifact_dir / "shared_container_run",
        executor="jupyter_notebook_shared",
    )
    shared_train_result = shared_pipeline.train(num_boost_round=60, random_state=42)
    print("Shared container TrainingResult:", shared_train_result.to_dict())
else:
    shared_train_result = None
    print("Shared Feast store is empty (expected in isolated offline tests).")

In [ ]:
def build_separable_banking_batch(
    *, num_customers: int = 160, num_apps: int = 480, seed: int = 23
) -> RawBankingBatch:
    rng = np.random.default_rng(seed)
    base_ts = datetime(2026, 1, 1, tzinfo=UTC)
    tiers = np.array([35_000.0, 75_000.0, 130_000.0])
    incomes = np.array([tiers[i % 3] + rng.uniform(-5000.0, 5000.0) for i in range(num_customers)])
    homes = np.array([HOME_OWNERSHIP_VALUES[i % 4] for i in range(num_customers)])
    cust_ids = [f"NB-CUST-{i:04d}" for i in range(num_customers)]
    cc_nums = [f"40009999{i:08d}" for i in range(num_customers)]

    profiles_df = pd.DataFrame(
        {
            "customer_id": cust_ids,
            "event_timestamp": pd.Series(
                pd.to_datetime([base_ts] * num_customers, utc=True).as_unit("ns")
            ),
            "age_years": rng.integers(23, 68, size=num_customers, dtype=np.int64),
            "income_annual": incomes.astype(np.float64),
            "employment_years": rng.uniform(1.0, 25.0, size=num_customers).astype(np.float64),
            "home_ownership": homes,
            "tenure_months": rng.integers(6, 120, size=num_customers, dtype=np.int64),
            "num_active_cards": np.ones(num_customers, dtype=np.int64),
        }
    )

    card_rows = []
    for idx in range(num_customers):
        for step in range(3):
            card_rows.append(
                {
                    "event_id": f"NB-EVT-{idx:04d}-{step}",
                    "cc_num": cc_nums[idx],
                    "customer_id": cust_ids[idx],
                    "event_timestamp": base_ts + timedelta(hours=step + 1),
                    "amount": float(np.round(rng.uniform(25.0, 300.0), 2)),
                    "latitude": 40.7,
                    "longitude": -74.0,
                }
            )
    card_events_df = pd.DataFrame(card_rows).loc[:, list(RAW_CARD_EVENTS_COLUMNS)]
    card_events_df["event_timestamp"] = pd.Series(
        pd.to_datetime(card_events_df["event_timestamp"], utc=True).dt.as_unit("ns")
    )

    app_cust_indices = np.arange(num_apps) % num_customers
    app_ts = [base_ts + timedelta(hours=6 * (i + 1)) for i in range(num_apps)]
    dti = rng.uniform(0.05, 0.75, size=num_apps)
    util = rng.uniform(0.05, 0.90, size=num_apps)
    delinq = rng.integers(0, 4, size=num_apps, dtype=np.int64)
    pub_rec = (rng.random(num_apps) < 0.25).astype(np.int64)
    risk_score = 2.6 * dti + 1.8 * util + 0.65 * delinq + 0.9 * pub_rec - 2.35
    labels = (risk_score > 0.0).astype(np.int64)
    flip_mask = rng.random(num_apps) < 0.02
    labels = np.where(flip_mask, 1 - labels, labels).astype(np.int64)

    apps_df = pd.DataFrame(
        {
            "application_id": [f"NB-APP-{i:05d}" for i in range(num_apps)],
            "customer_id": [cust_ids[idx] for idx in app_cust_indices],
            "event_timestamp": pd.Series(pd.to_datetime(app_ts, utc=True).as_unit("ns")),
            "loan_amount": rng.uniform(5_000.0, 60_000.0, size=num_apps).astype(np.float64),
            "loan_term_months": rng.choice([12, 24, 36, 48, 60], size=num_apps).astype(np.int64),
            "dti_ratio": dti.astype(np.float64),
            "existing_delinquencies": delinq,
            "utilization_ratio": util.astype(np.float64),
            "pub_rec_flag": pub_rec,
            "default_label": labels,
        }
    )
    return RawBankingBatch(
        customer_profiles=profiles_df,
        card_events=card_events_df,
        credit_applications=apps_df,
        batch_id="nb-stage1-backfill",
    )


sandbox_batch = build_separable_banking_batch()
feature_res = FeaturePipeline(feature_store=sandbox_store).run_batch(sandbox_batch)
print(
    "Sandbox FeaturePipeline rows_written:",
    {group: receipt.rows_written for group, receipt in feature_res.receipts.items()},
)

sandbox_registry = MlflowModelRegistry(
    tracking_uri=sandbox_tracking_uri,
    experiment_name=sandbox_experiment,
    environment="local",
    artifact_cache_dir=sandbox_artifact_dir / "mlflow_cache",
)
sandbox_pipeline = TrainingPipeline(
    feature_store=sandbox_store,
    model_registry=sandbox_registry,
    model_name=sandbox_model_name,
    artifact_dir=sandbox_artifact_dir / "sandbox_run",
    executor="jupyter_notebook",
)
sandbox_result = sandbox_pipeline.train(num_boost_round=50, random_state=42)

assert sandbox_result.convergence_passed is True
assert sandbox_result.parity_pass is True
assert sandbox_result.onnx_max_abs_diff is not None and sandbox_result.onnx_max_abs_diff < 1e-5
assert sandbox_result.fairness_passed is True
assert sandbox_result.serving_smoke_passed is True
assert sandbox_result.gate_passed is True
assert sandbox_result.promoted is True

champion = sandbox_registry.get_champion(sandbox_model_name)
assert champion is not None
first_version = champion.version
print(
    f"Promoted @champion v{champion.version} in MLflow: AUC={sandbox_result.holdout_auc:.3f}, "
    f"Recall={sandbox_result.holdout_recall:.3f}, Brier={sandbox_result.brier:.3f}, "
    f"ONNX diff={sandbox_result.onnx_max_abs_diff:.2e}"
)

In [ ]:
restored_paths = sandbox_registry.download_artifacts(
    champion, sandbox_artifact_dir / "restored_bundle"
)
reloaded_pp = TabularPreprocessor.load(restored_paths["preprocessor"])
onnx_sess = ort.InferenceSession(restored_paths["onnx"].read_bytes())

apps = sandbox_store.read_feature_group("credit_applications")
entity_df = apps.loc[:, ["application_id", "customer_id", "event_timestamp"]].tail(10)
inference_df = sandbox_store.get_historical_features(
    entity_df,
    ["customer_profiles", "card_window_aggs", "credit_applications"],
    exclude_labels=True,
)
X_infer = reloaded_pp.transform_features(inference_df)
onnx_outputs = onnx_sess.run(None, {"input": X_infer})
probs = np.asarray(onnx_outputs[1], dtype=np.float64)[:, 1]
onnx_scores_valid = bool(
    probs.shape == (len(entity_df),) and np.all((probs >= 0.0) & (probs <= 1.0))
)
assert onnx_scores_valid

preview = pd.DataFrame(
    {
        "application_id": inference_df["application_id"],
        "customer_id": inference_df["customer_id"],
        "default_probability": np.round(probs, 4),
        "predicted_default": (probs >= 0.5).astype(int),
        "model_version": champion.version,
        "feature_view_version": champion.feature_view_version,
    }
)
preview

In [ ]:
second_pipeline = TrainingPipeline(
    feature_store=sandbox_store,
    model_registry=sandbox_registry,
    model_name=sandbox_model_name,
    artifact_dir=sandbox_artifact_dir / "sandbox_run_2",
    gate_config=CreditRiskGateConfig(tie_tolerance=0.005),
    executor="jupyter_notebook",
)
second_result = second_pipeline.train(num_boost_round=50, random_state=42)
assert second_result.gate_passed is True
assert second_result.promoted is True
second_version = second_result.champion_version
assert second_version is not None and second_result.previous_version == first_version

active_champion = sandbox_registry.get_champion(sandbox_model_name)
prior_previous = sandbox_registry.get_previous(sandbox_model_name)
assert active_champion is not None and prior_previous is not None
rotation_verified = (
    active_champion.version == second_version and prior_previous.version == first_version
)

blocked_rollback = sandbox_registry.rollback_to_previous(
    sandbox_model_name, expected_feature_view_version="v99"
)
incompatible_rollback_blocked = blocked_rollback["ok"] is False

rollback_res = sandbox_registry.rollback_to_previous(sandbox_model_name)
restored_champion = sandbox_registry.get_champion(sandbox_model_name)
restored_previous = sandbox_registry.get_previous(sandbox_model_name)
rollback_verified = bool(
    rollback_res["ok"] is True
    and restored_champion is not None
    and restored_previous is not None
    and restored_champion.version == first_version
    and restored_previous.version == second_version
    and Path(restored_champion.onnx_model_path).exists()
)

assert rotation_verified
assert incompatible_rollback_blocked
assert rollback_verified
print(
    "Active @champion after rotation:",
    active_champion.version,
    "| After rollback @champion:",
    restored_champion.version,
    "| @previous:",
    restored_previous.version,
)
print("Lineage feature_schema_hash:", restored_champion.feature_schema_hash[:16] + "...")
print("Lineage dataset_sha256:", restored_champion.dataset_sha256[:16] + "...")